# This notebook can be used to calculate FID and DSC metrics

### Compute FID

In [ ]:
# Import
import torch
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd

from tqdm.notebook import tqdm
from src.util import metrics

In [ ]:
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

##### Example on RETOUCH data

In [ ]:
# Load DataFrames
clean_data_csv = pd.read_csv("../data/dataset_configs/retouch_original_spectralis_local.csv")
noisy_data_csv = pd.read_csv("../data/dataset_configs/retouch_original_cirrus_local.csv")

# Cache all volumes - NOTE: Make sure the data is properly scaled to [0, 1] here!
clean_volumes_list = [np.load(vol) for vol in tqdm(clean_data_csv["volume"], desc="Loading volumes")]
noisy_volumes_list = [np.load(vol) for vol in tqdm(noisy_data_csv["volume"], desc="Loading volumes")]

# Stack all volumes for proper 2D processing
clean_bscans    = np.concatenate(clean_volumes_list)
noisy_bscans    = np.concatenate(noisy_volumes_list)
print(f"Shape of clean bscans array: {clean_bscans.shape}\nShape of noisy bscans array: {noisy_bscans.shape}")

In [ ]:
# Compute embeddings    - NOTE: Might need log-in for hf_hub
clean_embeddings    = metrics.get_retfound_activations_from_array(images=clean_bscans, device=device, batch_size=32)
noisy_embeddings    = metrics.get_retfound_activations_from_array(images=noisy_bscans, device=device, batch_size=32)
print(f"Embeddings clean: {clean_embeddings.shape}\nEmbeddings noisy: {noisy_embeddings.shape}")

In [ ]:
# Compute FID
score_clean_vs_clean    = metrics.calculate_fid(act1=clean_embeddings, act2=clean_embeddings)
score_clean_vs_noisy    = metrics.calculate_fid(act1=clean_embeddings, act2=noisy_embeddings)
print(f"FID scores\nclean vs. clean: {score_clean_vs_clean:.2f}\nclean vs. noisy: {score_clean_vs_noisy:.2f}")

### Compute DSC

In [1]:
# Imports
import skimage

import pandas as pd
import numpy as np

from tqdm.notebook import tqdm

In [2]:
# Resize transform for the segmentation masks/annotations
resize = lambda x: np.stack([skimage.transform.resize(bscan, (224, 224), order=0) for bscan in x], axis=0)

In [3]:
# Computation Dice coefficient
def dice_coefficient_per_class(pred, gt, num_classes=4, smooth=1e-7):
    dice_scores = np.zeros(num_classes)
    
    for c in range(num_classes):

        # Create binary masks for current class
        pred_c  = (pred == c)
        gt_c    = (gt == c)
        
        # Calculate intersection and union
        intersection    = np.sum(pred_c * gt_c)
        pred_sum        = np.sum(pred_c)
        gt_sum          = np.sum(gt_c)
        
        # Calculate Dice coefficient
        if gt_sum == 0:
            dice = np.nan
        else:
            dice = (2.0 * intersection) / (pred_sum + gt_sum + smooth)
        dice_scores[c] = dice
    
    return dice_scores

#### Example on RETOUCH data
(This does not make much sense of course, because the data is not paired. Still, this is just an example.)

In [4]:
# Load DataFrames
clean_data_csv = pd.read_csv("../data/dataset_configs/retouch_original_spectralis_local.csv")
noisy_data_csv = pd.read_csv("../data/dataset_configs/retouch_original_cirrus_local.csv")

# Cache all segmentations
clean_seg_list = [np.load(vol) for vol in tqdm(clean_data_csv["mask"], desc="Loading volumes")]
noisy_seg_list = [np.load(vol) for vol in tqdm(noisy_data_csv["mask"], desc="Loading volumes")]

# Resize for downstream evaluation
clean_seg_list_resized = [resize(mask) for mask in clean_seg_list]
noisy_seg_list_resized = [resize(mask) for mask in noisy_seg_list]

Loading volumes:   0%|          | 0/24 [00:00<?, ?it/s]

Loading volumes:   0%|          | 0/24 [00:00<?, ?it/s]

In [5]:
# Clean vs. Clean
dice_coeffs_clean_clean = []

# Iterate all cases
for pred, gt in tqdm(zip(clean_seg_list_resized, clean_seg_list_resized), total=len(clean_seg_list_resized), desc="Processing segmentations", leave=False):
    dc = dice_coefficient_per_class(pred=pred, gt=gt)
    dice_coeffs_clean_clean.append(dc[1:])  # Omit the background class @ idx=0

# Dice scores for each case and each class - shape (#cases, #classes)
dcs_clean_clean = np.stack(dice_coeffs_clean_clean, axis=0)

# Aggregate per class over all cases
dcs_clean_clean_per_class = np.nanmean(dcs_clean_clean, axis=0)
print(f"Average dice score per class: {dcs_clean_clean_per_class}, Mean over classes: {dcs_clean_clean_per_class.mean():.3f}")

Processing segmentations:   0%|          | 0/24 [00:00<?, ?it/s]

Average dice score per class: [1. 1. 1.], Mean over classes: 1.000


In [6]:
# Noisy vs. Noisy
dice_coeffs_noisy_noisy = []

# Iterate all cases
for pred, gt in tqdm(zip(noisy_seg_list_resized, noisy_seg_list_resized), total=len(clean_seg_list_resized), desc="Processing segmentations", leave=False):
    dc = dice_coefficient_per_class(pred=pred, gt=gt)
    dice_coeffs_noisy_noisy.append(dc[1:])  # Omit the background class @ idx=0

# Dice scores for each case and each class - shape (#cases, #classes)
dcs_noisy_noisy = np.stack(dice_coeffs_noisy_noisy, axis=0)

# Aggregate per class over all cases
dcs_noisy_noisy_per_class = np.nanmean(dcs_noisy_noisy, axis=0)
print(f"Average dice score per class: {dcs_noisy_noisy_per_class}, Mean over classes: {dcs_noisy_noisy_per_class.mean():.3f}")

Processing segmentations:   0%|          | 0/24 [00:00<?, ?it/s]

Average dice score per class: [1. 1. 1.], Mean over classes: 1.000
